# 00_로컬환경설정

In [1]:
# 라이브러리 호출
import re 
import pandas as pd
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime

from helper import comenv
cfg = comenv.setup(
    seed=42,
    cpu_ratio=0.6,
    tokenizers_parallel=True,
    float32_precision="high",
    tf32=True,
    cudnn_benchmark=True,
    suppress_warnings=True,
    verbose=True,
    force_utf8=True,
    pandas_display=True,
    pd_max_columns=50,
    pd_max_rows=100,
    pd_max_colwidth=80,
    matplotlib_backend="auto",
    debug_cuda=False,
)

Platform             : Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.35
Working directory    : /d/git/01_mdls_ds8_open/10_rs/260409_프로젝트
Python version       : 3.10.14
Python executable    : /opt/conda/bin/python
Python prefix        : /opt/conda
site-packages        : ['/opt/conda/lib/python3.10/site-packages']
Encoding (stdout)    : UTF-8
----------------------------------------------------------------------
PyTorch              : 2.3.1
CUDA available       : True
Device               : cuda
Random Seed          : 42
CPU count            : 20
CPU threads (used)   : 12  (ratio=0.6)
Korean font          : NanumGothic
----------------------------------------------------------------------
GPU name             : NVIDIA GeForce RTX 5060 Ti
GPU count            : 1
GPU memory (total)   : 15.93 GB
GPU memory (free)    : 14.80 GB
TF32                 : True
cuDNN benchmark      : True
cuDNN deterministic  : True
FP32 precision       : high
CUDA (torch built)   : 12.1
CUDA (system 

# 01_데이터 불러오기

In [2]:
# README.txt에 나와있는 컬럼 정보를 지정
user_colums = ['user_id', 'gender', 'age', 'occupation', 'zip']
rating_columns = ['user_id', 'movie_id', 'rating', 'timestamp']
movie_columns = ['movie_id', 'title', 'genres']

# data 경로 설정 
data_path = './data'

# 데이터 불러오기 
users = pd.read_csv(f'{data_path}/users.dat', sep='::', header=None, names=user_colums, engine='python')
ratings = pd.read_csv(f'{data_path}/ratings.dat', sep='::', header=None, names=rating_columns, engine='python')
movies = pd.read_csv(f'{data_path}/movies.dat', sep='::', header=None, names=movie_columns, engine='python', encoding='latin-1')

In [3]:
# 사용자 데이터 확인
print(users.shape)
users.head()

(6040, 5)


,user_id,gender,age,occupation,zip
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455


In [4]:
# 영화 데이터 확인
print(movies.shape)
movies.head()

(3883, 3)


,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [5]:
# 평점 데이터 확인
print(ratings.shape)
ratings.head()

(1000209, 4)


,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


# 02_영화데이터 전처리

In [6]:
# 영화 데이터(movies)를 전처리해 봅시다.
# 이전에는 영화 제목과 연도를 분리하는 과정만 진행했지만 
# 이번에는 한 가지의 데이터를 더 추출해보려고 합니다.

# 바로 '년대'인데요, 예를 들어, 1995년에 개봉한 영화는 '90년대' 영화라고도 많이 말하죠. 
# '년대' 정보를 뽑아내는 과정을 추가하려고 합니다.

# 제목 뒤에 붙어 있는 연도를 정규표현식을 활용해 추출합니다. 
movies['movie_year'] = movies['title'].str.extract(r'\((\d{4})\)')
movies.head()

,movie_id,title,genres,movie_year
0,1,Toy Story (1995),Animation|Children's|Comedy,1995
1,2,Jumanji (1995),Adventure|Children's|Fantasy,1995
2,3,Grumpier Old Men (1995),Comedy|Romance,1995
3,4,Waiting to Exhale (1995),Comedy|Drama,1995
4,5,Father of the Bride Part II (1995),Comedy,1995


In [7]:
# 년대를 뽑아내는 과정
# 제목에서 영화의 연도를 뽑아낸다. -> 연도에서 연도를 10으로 나눈 나머지를 빼면 년대이다.
# 예) 1995년 영화
# 1995 % 10 = 5
# 1995 - 5 = 1990 ➔ 90년대 영화
# 이를 코드로 구현하면 아래와 같습니다.

a = 1995 % 10
b = 1995 - a

print(a)
print(b)

5
1990


In [8]:
# 90년대 영화, 2000년대 영화 등과 같은 년대 정보를 추출합니다. 
movies['movie_decade'] = movies['title'].str.extract(r'\((\d{4})\)')[0].astype(int).apply(lambda x: str(x - (x % 10)) + 's')

# title 컬럼에서 연도 정보를 빼고 제목만 따로 추출합니다.
movies['title'] = movies['title'].apply(lambda x: re.sub(r'\s*\(\d{4}\)', '', x))

In [9]:
movies.head()

,movie_id,title,genres,movie_year,movie_decade
0,1,Toy Story,Animation|Children's|Comedy,1995,1990s
1,2,Jumanji,Adventure|Children's|Fantasy,1995,1990s
2,3,Grumpier Old Men,Comedy|Romance,1995,1990s
3,4,Waiting to Exhale,Comedy|Drama,1995,1990s
4,5,Father of the Bride Part II,Comedy,1995,1990s


In [10]:
# 이전에 했던 것과 같이 장르를 분리해서 추출합니다. 
# 다만 이전에는 분리된 장르를 행(row)에 추가하였다면, 
# 이번에는 컬럼(열)에 추가해 두겠습니다. 
# 이렇게 구성하는 이유는 나중에 모델 input에 넣을 때 편리하기 때문입니다.


# | 기호를 기준으로 장르 데이터를 분리합니다. 
genres_split = movies['genres'].str.split('|')

max_genres = genres_split.apply(len).max() 

# 각 장르별로 새로운 컬럼을 생성합니다.
for i in range(max_genres):
    movies[f'genre{i + 1}'] = genres_split.apply(lambda x: x[i] if i < len(x) else None)

# 원래 있던 장르 컬럼은 날려줍니다.
movies.drop('genres', axis=1, inplace=True)

# None은 공백으로 채웁니다.
movies.fillna('', inplace=True)

print(movies.shape)
movies.head()


(3883, 10)


,movie_id,title,movie_year,movie_decade,genre1,genre2,genre3,genre4,genre5,genre6
0,1,Toy Story,1995,1990s,Animation,Children's,Comedy,,,
1,2,Jumanji,1995,1990s,Adventure,Children's,Fantasy,,,
2,3,Grumpier Old Men,1995,1990s,Comedy,Romance,,,,
3,4,Waiting to Exhale,1995,1990s,Comedy,Drama,,,,
4,5,Father of the Bride Part II,1995,1990s,Comedy,,,,,


In [11]:
# 모든 장르의 종류는 아래와 같습니다.
set(movies['genre1'].unique().tolist() + movies['genre2'].unique().tolist() + movies['genre3'].unique().tolist())

{'',
 'Action',
 'Adventure',
 'Animation',
 "Children's",
 'Comedy',
 'Crime',
 'Documentary',
 'Drama',
 'Fantasy',
 'Film-Noir',
 'Horror',
 'Musical',
 'Mystery',
 'Romance',
 'Sci-Fi',
 'Thriller',
 'War',
 'Western'}

# 03_평점 데이터 전처리 

In [ ]:
# 평점 데이터(ratings)도 조금 더 세분화하여 쪼갤 수 있습니다. 
# 바로 timestmap를 이용하면 되는데요.

# 앞서 ratings에 있는 timestamp는 Unix timestamp라고 언급하였습니다. 
# Unix timestamp란 1970년 1월 1일 00:00:00 UTC 이후 경과한 '초' 수를 말합니다. 
# 따라서 '연-월-일' 로 변환이 필요합니다.

# '년-월-일'로 변환하는 방법은 간단합니다. 
# Unix timestamp는 워낙 많이 사용하기 때문에 이미 파이썬 내장 함수에도 기능이 존재기 때문이죠. 
# datetime의 fromtimestamp 함수를 사용하면 바로 변환할 수 있습니다.

# 예를 살펴 볼까요?

timestamp = 978300760 
dt_object = datetime.fromtimestamp(timestamp)

print(dt_object.strftime("%Y-%m-%d"))

2000-12-31


In [13]:
# 평점 데이터 전체에 적용해 봅시다.
ratings['timestamp'] = ratings['timestamp'].apply(lambda x : datetime.fromtimestamp(x).strftime("%Y-%m-%d"))
ratings.head()

,user_id,movie_id,rating,timestamp
0,1,1193,5,2000-12-31
1,1,661,3,2000-12-31
2,1,914,3,2000-12-31
3,1,3408,4,2000-12-31
4,1,2355,5,2001-01-06


In [14]:
# 변환된 '년-월-일' 데이터를 활용해서 년도, 월, 그리고 년대를 뽑아냅시다.
ratings['rating_year'] = ratings['timestamp'].apply(lambda x : x.split("-")[0]) 
ratings['rating_month'] = ratings['timestamp'].apply(lambda x : x.split("-")[1])
ratings['rating_decade'] = ratings['rating_year'].astype(int).apply(lambda x: str(x - (x % 10)) + 's')

In [15]:
ratings.head()

,user_id,movie_id,rating,timestamp,rating_year,rating_month,rating_decade
0,1,1193,5,2000-12-31,2000,12,2000s
1,1,661,3,2000-12-31,2000,12,2000s
2,1,914,3,2000-12-31,2000,12,2000s
3,1,3408,4,2000-12-31,2000,12,2000s
4,1,2355,5,2001-01-06,2001,01,2000s


In [16]:
# 이렇게 뽑아낸 데이터들은 임시로 저장을 해두겠습니다. 
# 임시 저장은 필수는 아니지만, 편의를 위해 저장해두려고 합니다.
# 저장된 데이터들은 _prepro라는 이름을 붙여서 csv 형태로 저장해 두겠습니다.

# 임시 저장 
movies.to_csv(f"{data_path}/movies_prepro.csv", index=False)
ratings.to_csv(f"{data_path}/ratings_prepro.csv", index=False)
users.to_csv(f"{data_path}/users_prepro.csv", index=False)

# 04_모델 입력 데이터 생성

- 추천 시스템에 활용되는 데이터는 크게 아래와 같이 구분될 수 있습니다.
   - 명시적 데이터(Explicit data): 사용자가 직접적으로 선호도를 표현한 데이터(예: 평점, 구독, 댓글, 리뷰, 좋아요, 싫어요, 차단 등)
   - 암묵적 데이터(Implicit data): 사용자가 간접적으로 선호도를 표현한 데이터(예: 클릭 여부, 검색 기록, 방문 페이지, 마우스 움직임, 구매 내역, 시청 시간대 등)

- 명시적 데이터, 예를 들어 평점 데이터라면 1~5점과 같은 분포로 데이터가 존재하게 되고 이는 이진 분류(binary classification) 문제라기 보다 회귀(regression) 문제나 다중 분류(multiclass classification) 문제라고 볼 수 있을 것입니다.
- 하지만 암묵적 데이터인 클릭 여부로 보면 어떨까요?
   - 클릭 여부는 '클릭을 했다, 안했다'와 같이 0과 1로 분리할 수 있습니다. 이 때 클릭을 했다는 것은 '선호'한다라는 의미도 담고 있기에, 선호 정보를 모델링한다고도 볼 수 있습니다.

- 무엇이 정답인지는 따로 정해진 바가 없습니다. 데이터와 서비스 전략에 따라 정답은 달라집니다.
- 이번 프로젝트에서는 '선호했다'를 기준으로 살펴보려고 합니다.
   - 그런데 여기서 문제가 하나 있습니다. 그럼 '선호했다'라는 정보는 무엇이고 '선호하지 않는다'라는 정보는 무엇일까요? 그리고 그런 데이터가 MovieLens에 있을까요? 이 문제를 풀어나가보려고 합니다.

## 04-1_랜덤 샘플링 기반 방법

- 우리가 가지고 있는 데이터에서 '선호도'를 뽑아내기 위한 데이터로는 평점 데이터(ratings)가 있습니다.
- 문제는 평점이 1부터 5점까지라서 '어떤 것을 선호한다'라고 말하기 어렵습니다. 
- 그래서 여기서부터는 가설을 세우고 진행하는 것이 좋습니다.

- 우리는 먼저 랜덤 샘플링 기반 방법으로 '선호', '비선호'를 추출할 겁니다. 순서는 아래와 같습니다.

- 선호 데이터를 추출한다.
   - 사용자가 3점 이상의 점수를 부여한 영화를 '선호'한다고 가정한다.
   - 이 데이터는 label=1인 값이다.
- 비선호 데이터를 추출한다.
   - 사용자가 선호했던 영화 리스트를 추출한다
   - 전체 영화 중 사용자가 선호한다고 체크하지 않은 영화 리스트를 추출한다. 
      - 만약 전체 영화가 100,사용자가 선호한다고 한 영화가 10이면, 선호하지 않은 영화는 90
   - 영화 리스트 중 일부를 랜덤으로 샘플링한다. 이때 선호 영화 1개당 5개의 비선호 영화를 추출한다.
      - 예를 들어 선호 영화가 10 이면 비선호 영화는 총 50개를 추출 즉 비선호 영화 90개 중 50개를 추출
   - 이 비선호 영화를 label=0으로 세팅한다.

In [17]:
# 1. 3점 이상의 점수를 부여한 영화를 '선호'영화라고 가정하고, 이를 label=1로 생성합니다.
ratings = ratings[ratings['rating'] >= 3]
ratings['label'] = 1
ratings.drop('rating', axis=1, inplace=True)
print(ratings.shape)
ratings.head()

(836478, 7)


,user_id,movie_id,timestamp,rating_year,rating_month,rating_decade,label
0,1,1193,2000-12-31,2000,12,2000s,1
1,1,661,2000-12-31,2000,12,2000s,1
2,1,914,2000-12-31,2000,12,2000s,1
3,1,3408,2000-12-31,2000,12,2000s,1
4,1,2355,2001-01-06,2001,01,2000s,1


In [18]:
# 1. 사용자가 봤던(선호했던) 영화 리스트를 추출합니다. 사용자마다 평점이 3점 이상인 영화 리스트가 구성됩니다. 
user_seen_movies = ratings.groupby('user_id')['movie_id'].apply(list).reset_index()
user_seen_movies.head()

,user_id,movie_id
0,1,"[1193, 661, 914, 3408, 2355, 1197, 1287, 2804, 594, 919, 595, 938, 2398, 291..."
1,2,"[1357, 3068, 1537, 647, 2194, 648, 2268, 2628, 1103, 2916, 3468, 1210, 1792,..."
2,3,"[3421, 648, 1394, 3534, 104, 2735, 1210, 1431, 3868, 1079, 2997, 1615, 1291,..."
3,4,"[3468, 1210, 2951, 1214, 1036, 260, 2028, 480, 1198, 1954, 1097, 3418, 3702,..."
4,5,"[2987, 2333, 1175, 39, 2337, 1535, 1392, 1466, 1683, 866, 1684, 2770, 215, 1..."


In [ ]:
import random

# 2. 먼저 고유 영화와 고유 사용자들을 가지고 옵니다. 
unique_movies = movies['movie_id'].unique()
unique_users = users['user_id'].unique()
negative_users = []
negative_movies = []
negative_labels = []

# 사용자별로 하나씩 진행합니다.
for user in unique_users:
    # 충분한 이력이 없는 사용자는 넘어갑니다. 충분한 이력이 없는 사용자 데이터는 훈련에 방해가 되고 overfitting 등이 될 수 있습니다.
    if len(user_seen_movies[user_seen_movies['user_id'] == user]) < 1:
        continue
    # 2-1. 해당 사용자가 선호하는 영화 리스트를 가지고 옵니다.
    user_seen_movie_list = user_seen_movies[user_seen_movies['user_id'] == user]['movie_id'].values[0]
    # 2-2. 전체 영화 중 사용자가 선호한 영화 정보를 제외합니다.
    user_non_seen_movie_list = list(set(unique_movies) - set(user_seen_movie_list))
    # 2-3. 선호 영화 1개당 비선호 영화 5개를 추출합니다. 
    sample_pop_size = len(user_seen_movie_list)*5
    # 만약 비선호 영화 샘플 개수가 전체 영화 개수보다 크면, 비선호 영화 샘플 개수는 전체 영화 개수에서 사용자가 선호한 영화의 개수를 뺀 값(해당 사용자 입장에선 샘플링 최대 값)으로 설정합니다.
    if len(unique_movies) - len(user_seen_movie_list) < len(user_seen_movie_list)*5 :
        sample_pop_size = len(unique_movies) - len(user_seen_movie_list)
    # 랜덤으로 추출합니다. 
    user_negative_movie_list = random.sample(user_non_seen_movie_list, sample_pop_size)
    
    # 해당 값들을 리스트에 저장합니다.
    negative_users += [user for _ in range(len(user_negative_movie_list))]
    negative_movies += user_negative_movie_list
    negative_labels += [0 for _ in range(len(user_negative_movie_list))]

In [21]:
print(len(negative_users))
print(len(negative_movies))
print(len(negative_labels))

4068124
4068124
4068124


In [22]:
negative_ratings_df = pd.DataFrame({'user_id' : negative_users, 'movie_id' : negative_movies, 'label':negative_labels})
print(negative_ratings_df.shape)
negative_ratings_df.head()

(4068124, 3)


,user_id,movie_id,label
0,1,2732,0
1,1,464,0
2,1,106,0
3,1,3157,0
4,1,1164,0


In [ ]:
# 최종적으로 모델 훈련에 사용하는 데이터를 구성하기 위해서 필요한 컬럼만 추출해 활용합니다. 
# 필요한 컬럼은 아래와 같습니다.
# 평점 : 사용자 ID, 영화 ID, 레이블(label)
# 영화 : 영화 ID, 년대, 년도, 장르1
# 사용자 : 사용자 ID, 성별, 나이, 지역, 직업
# 이 데이터를 모아 하나로 합쳐줍니다.

In [23]:
ratings_df = ratings[['user_id', 'movie_id', 'label']] 
ratings_df = pd.concat([ratings_df, negative_ratings_df], axis=0)
movies_df = movies[['movie_id', 'movie_decade', 'movie_year', 'genre1']]
movies_df.columns = ['movie_id', 'decade', 'movie_year', 'genre']
user_df = users[['user_id', 'gender', 'age', 'occupation', 'zip']]

In [24]:
merge_mlens_data = pd.merge(ratings_df, movies_df, on='movie_id')
merge_mlens_data = pd.merge(merge_mlens_data, user_df, on='user_id')
merge_mlens_data.dropna(inplace=True)
print(merge_mlens_data.shape)
merge_mlens_data.head()

(4904602, 10)


,user_id,movie_id,label,decade,movie_year,genre,gender,age,occupation,zip
0,1,1193,1,1970s,1975,Drama,F,1,10,48067
1,1,661,1,1990s,1996,Animation,F,1,10,48067
2,1,914,1,1960s,1964,Musical,F,1,10,48067
3,1,3408,1,2000s,2000,Drama,F,1,10,48067
4,1,2355,1,1990s,1998,Animation,F,1,10,48067


In [25]:
merge_mlens_data = merge_mlens_data[['user_id', 'movie_id','decade', 'movie_year', 'genre', 'gender', 'age', 'occupation', 'zip', 'label']]
print(merge_mlens_data.shape)
merge_mlens_data.head()

(4904602, 10)


,user_id,movie_id,decade,movie_year,genre,gender,age,occupation,zip,label
0,1,1193,1970s,1975,Drama,F,1,10,48067,1
1,1,661,1990s,1996,Animation,F,1,10,48067,1
2,1,914,1960s,1964,Musical,F,1,10,48067,1
3,1,3408,2000s,2000,Drama,F,1,10,48067,1
4,1,2355,1990s,1998,Animation,F,1,10,48067,1


In [26]:
# 데이터를 하나로 합치면 위와 같은 형태의 데이터프레임이 구성됩니다. 
# 이 데이터를 활용하여 모델을 훈련할 수 있습니다. 
# 이 데이터를 movielens_rcmm_v1이라고 명명하고 csv 형식으로 저장해 두겠습니다.
merge_mlens_data.to_csv(f'{data_path}/movielens_rcmm_v1.csv', index=False)

## 04-2_선호도로 나누기


- 위에서 진행한 '1. 랜덤 샘플링 기반 방법'은 그리 좋은 방법은 아닙니다. 
- 왜냐하면 사용자 정보를 무작위로 샘플링을 해서 임의로 데이터를 생성했기 때문입니다. 
- 가장 좋은 것은 원본 데이터를 활용하는 것이죠.

- 따라서 2번째 과정에서는 샘플링을 하지 않고 데이터를 선호도로 나누려고 합니다. 
- 이후 모델을 훈련할 때도 이 데이터를 활용할 것입니다.

- 먼저 데이터 전처리를 하고 저장했던 데이터를 불러옵시다.

In [27]:
users_df = pd.read_csv(f'{data_path}/users_prepro.csv')
ratings_df = pd.read_csv(f'{data_path}/ratings_prepro.csv')
movies_df = pd.read_csv(f'{data_path}/movies_prepro.csv')

In [28]:
print(users_df.columns)
print(ratings_df.columns)
print(movies_df.columns)

Index(['user_id', 'gender', 'age', 'occupation', 'zip'], dtype='object')
Index(['user_id', 'movie_id', 'rating', 'timestamp', 'rating_year',
       'rating_month', 'rating_decade'],
      dtype='object')
Index(['movie_id', 'title', 'movie_year', 'movie_decade', 'genre1', 'genre2',
       'genre3', 'genre4', 'genre5', 'genre6'],
      dtype='object')


In [ ]:
# 1번 과정에서는 평점이 3점 이상인 데이터를 label=1로 설정하고 label=0은 랜덤 샘플링으로 추출
# 이번 과정에서는 다음과 같은 순서로 데이터 선호도를 구성

# 평점이 4점 이상인 데이터를 label=1, 나머지를 label=0
# 필요한 데이터만 추출하고 저장

In [29]:
# 1. 4점 이상인 데이터를 1로, 아닌 데이터를 0으로 설정합니다.  
ratings_df['label'] = ratings_df['rating'].apply(lambda x : x >=4).astype(int)

ratings_df = ratings_df[['user_id', 'movie_id', 'rating_year','rating_month', 'rating_decade', 'label']]
ratings_df.head()

,user_id,movie_id,rating_year,rating_month,rating_decade,label
0,1,1193,2000,12,2000s,1
1,1,661,2000,12,2000s,0
2,1,914,2000,12,2000s,0
3,1,3408,2000,12,2000s,1
4,1,2355,2001,1,2000s,1


In [30]:
# 2. 필요 데이터만 가지고 옵니다. 특히 장르는 3개만 가지고 와서 활용합니다.  
movies_df = movies_df[['movie_id', 'movie_decade', 'movie_year', 'genre1', 'genre2', 'genre3']]
users_df = users_df[['user_id', 'gender', 'age', 'occupation', 'zip']]

In [31]:
# 필요한 데이터를 합쳐줍니다.   
merge_mlens_data = pd.merge(ratings_df, movies_df, on='movie_id')
merge_mlens_data = pd.merge(merge_mlens_data, users_df, on='user_id')
merge_mlens_data.fillna('no', inplace=True)
print(merge_mlens_data.shape)
merge_mlens_data.head()

(1000209, 15)


,user_id,movie_id,rating_year,rating_month,rating_decade,label,movie_decade,movie_year,genre1,genre2,genre3,gender,age,occupation,zip
0,1,1193,2000,12,2000s,1,1970s,1975,Drama,no,no,F,1,10,48067
1,1,661,2000,12,2000s,0,1990s,1996,Animation,Children's,Musical,F,1,10,48067
2,1,914,2000,12,2000s,0,1960s,1964,Musical,Romance,no,F,1,10,48067
3,1,3408,2000,12,2000s,1,2000s,2000,Drama,no,no,F,1,10,48067
4,1,2355,2001,1,2000s,1,1990s,1998,Animation,Children's,Comedy,F,1,10,48067


In [32]:
merge_mlens_data = merge_mlens_data[['user_id', 'movie_id','movie_decade', 'movie_year', 'rating_year', 'rating_month', 'rating_decade', 'genre1','genre2', 'genre3', 'gender', 'age', 'occupation', 'zip', 'label']]
print(merge_mlens_data.shape)
merge_mlens_data.head()

(1000209, 15)


,user_id,movie_id,movie_decade,movie_year,rating_year,rating_month,rating_decade,genre1,genre2,genre3,gender,age,occupation,zip,label
0,1,1193,1970s,1975,2000,12,2000s,Drama,no,no,F,1,10,48067,1
1,1,661,1990s,1996,2000,12,2000s,Animation,Children's,Musical,F,1,10,48067,0
2,1,914,1960s,1964,2000,12,2000s,Musical,Romance,no,F,1,10,48067,0
3,1,3408,2000s,2000,2000,12,2000s,Drama,no,no,F,1,10,48067,1
4,1,2355,1990s,1998,2001,1,2000s,Animation,Children's,Comedy,F,1,10,48067,1


In [33]:
# 저장합니다.
merge_mlens_data.to_csv(f'{data_path}/movielens_rcmm_v2.csv', index=False)